# M1.1 · 数据治理与 Point-in-Time（PIT）

> 量化研究的可信度建立在数据之上。错误的数据会让回测产生虚假收益，且难以察觉。
> 本节理解 PIT、幸存者偏差、前视偏差、复权、停牌、退市等数据治理问题，并在真实数据上做质量检查。

## 一、什么是 Point-in-Time（PIT）

PIT 要求：在历史任意时刻 t，只能使用 **t 时刻实际已经可得** 的信息。
- 财报有发布延迟：季度末的数据，实际几周后才公布；若用"期末日期"对齐就偷看了未来；
- 指数成分、行业分类会随时间调整，要用当时的成分而非现在的；
- 数据修正：基本面数据可能被修订，要保留当时版本。

> 违反 PIT 是最隐蔽的前视偏差来源，会让回测虚高。

## 二、加载真实数据并做质量检查

In [1]:
import sys, numpy as np, pandas as pd
sys.path.insert(0, r'C:\Users\gucao\Desktop\Qlib')
import qlib
from qlib.data import D
from l2.config import load_universe
import config
qlib.init(provider_uri=str(config.QLIB_DAILY_DIR), joblib_backend='threading')
codes = load_universe('hs300')
fields = ['$open','$close','$high','$low','$volume','$factor',
          '$mkt_cap_ard','$pe_ttm','$pb_lf']
df = D.features(codes, fields, start_time='2018-01-01', freq='day')
print('面板形状:', df.shape)
df.head(3)

[5096:MainThread](2026-11-05 16:34:15,788) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[5096:MainThread](2026-11-05 16:34:17,917) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[5096:MainThread](2026-11-05 16:34:17,918) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': WindowsPath('C:/Users/gucao/Desktop/Qlib/data/qlib_daily')}


面板形状: (568211, 9)


$open      $close       $high        $low  \
instrument datetime                                                     
SH600000   2018-01-02  153.860306  155.202469  155.812546  153.738297   
           2018-01-03  155.324478  154.470383  156.178589  154.470383   
           2018-01-04  154.958435  154.470383  155.324478  153.982330   

                          $volume    $factor  $mkt_cap_ard   $pe_ttm    $pb_lf  
instrument datetime                                                             
SH600000   2018-01-02  31323054.0  12.201452  3.733585e+11  6.874707  0.963660  
           2018-01-03  37839100.0  12.201452  3.715973e+11  6.842279  0.959114  
           2018-01-04  27883804.0  12.201452  3.715973e+11  6.842279  0.959114

In [2]:
# 缺失值检查：每个字段的 NaN 数量
miss = df.isna().sum()
print(miss.to_string())

$open           298
$close            0
$high           298
$low            298
$volume         298
$factor           0
$mkt_cap_ard      0
$pe_ttm           0
$pb_lf            0


## 三、停牌识别

A 股停牌时无成交，常见表现为成交量为 0 或价格缺失。

In [3]:
# 以成交量为 0 或收盘价缺失识别停牌
halt = df[(df['$volume'].fillna(0) == 0) | (df['$close'].isna())]
print('疑似停牌记录数:', len(halt), ' 占比 %.3f%%' % (100*len(halt)/len(df)))
# 单只股票停牌示例
sample = codes[0]
sub = df.xs(sample, level='instrument')
halt_days = sub[(sub['$volume'].fillna(0) == 0) | (sub['$close'].isna())]
print(sample, '停牌天数:', len(halt_days))

疑似停牌记录数: 2060  占比 0.363%
SZ000001 停牌天数: 0


## 四、复权处理

- 分红送股会让价格"跳空"，但不是真实亏损；
- 前复权/后复权用于让价格序列连续；Qlib 用复权因子 `$factor` 调整；
- 回测收益必须基于复权价，否则分红除权会被误判为下跌。

In [4]:
# 验证复权因子：后复权价 = 原始价 × 复权因子
sub = df.xs(codes[0], level='instrument').dropna(subset=['$close'])
adj_close = sub['$close'] * sub['$factor']
print('原始收盘价范围: %.2f ~ %.2f' % (sub['$close'].min(), sub['$close'].max()))
print('后复权价范围:   %.2f ~ %.2f' % (adj_close.min(), adj_close.max()))
print('复权因子非空且>0:', bool((sub['$factor'] > 0).all()))

原始收盘价范围: 775.65 ~ 1124.64
后复权价范围:   66185.54 ~ 95965.38
复权因子非空且>0: True


## 五、幸存者偏差与退市股

- 只用"今天还活着"的股票做历史回测，会系统性剔除已退市（往往表现差）的公司，导致收益虚高，即幸存者偏差；
- 正确做法：股票池包含历史上存在过、后来退市的股票；
- 退市股数据工程（占位）：
  1. 从 Wind 拉取已退市股票列表与退市日期；
  2. 保留其退市前的完整行情，纳入历史股票池；
  3. 回测在退市日按最后价格清算；
- 真实退市数据待确认 Wind 可拉取后落地（M1.1 先给方法与接口占位）。

In [5]:
# 接口占位：退市股加载（数据就绪后实现）
def load_delisted_codes():
    """返回历史退市股票及其退市日期；当前数据未就绪，返回空列表占位。"""
    return []   # TODO: 接入 Wind 退市数据后补全
print('当前退市股数据（占位）:', load_delisted_codes())

当前退市股数据（占位）: []


## 六、前视偏差与数据对齐

- 所有特征在用于预测时，必须确认其在 t 日收盘后才可得，预测 t+1；
- 宏观/基本面用发布时间而非所属期对齐；
- 滚动窗口只用窗口内数据，标准化用截面或滚动口径，避免全样本统计量泄漏。

In [6]:
# 演示正确的滞后对齐：用 t 日因子预测 t+1 收益
close = df['$close'].unstack('instrument')
ret_next = close.pct_change().shift(-1)   # 下一期收益（标签）
factor_t = close.pct_change(20)          # t 日可得的20日反转因子
aligned = pd.DataFrame({'factor': factor_t.stack(), 'label': ret_next.stack()}).dropna()
print('对齐后样本:', aligned.shape)
print(aligned.tail(2))

对齐后样本: (561911, 2)
                         factor     label
datetime   instrument                    
2026-10-30 SZ301308   -0.360491  0.011595
           SZ302132    0.082272  0.013200


## 七、数据质量校验清单

| 检查项 | 要求 |
|---|---|
| 缺失 | 明确缺失原因（停牌/数据缺口），有处理规则 |
| 日历 | 统一交易日历，非交易日无数据 |
| 复权 | 价格已复权，因子连续 |
| 极值 | 异常值/错误数据识别与处理 |
| PIT | 无未来信息，发布时间对齐 |
| 股票池 | 含退市股，避免幸存者偏差 |
| 一致性 | 跨字段、跨来源口径一致并对账 |

## 八、小结与面试要点

- PIT 保证只用当时可得信息；幸存者偏差、前视偏差、复权错误是回测虚高的三大元凶。
- 停牌用成交量/价格缺失识别；退市股需纳入历史池（当前为占位）。
- 数据治理是严谨研究的第一道质量门。

**面试高频：**
- 什么是 PIT，违反会怎样？（答：只用当时已公布信息，否则前视偏差、回测虚高）
- 什么是幸存者偏差？（答：剔除退市股导致收益虚高，需纳入历史全部股票）
- 复权为什么重要？（答：分红除权不是真实亏损，回测要用复权价）
- 怎么识别停牌？（答：成交量为0或价格缺失）

## 动手练习
1. 检查你数据里每只股票的停牌天数分布；
2. 写出退市股数据工程的完整步骤；
3. 举出一个可能产生前视偏差的具体场景并说明如何避免。

---
下一节 **M1.2 A 股交易机制**。